# Which chemicals are linked to thyroid-related AOPs?

We want to find chemical stressors connected to thyroid-related adverse outcome pathways (AOPs), inspect the annotations and chemical identifiers, and see which other key events their pathways contain.

We will start with titles containing **thyroid**, then follow the links recorded in [AOPWiki RDF](https://aopwiki.rdf.bigcat-bioinformatics.org/). This starting search will miss synonyms. A shared pathway is an association, not proof that a chemical directly causes every event.

Run the cells in order with this checkout of rdfsolve, pandas, and ipykernel installed. Requests run one at a time.

## Open the dataset

These lines read the available record types and fields, then generate a typed Python API. We skip counts and examples because this investigation does not need them. Query recording starts before mining.

In [1]:
import logging
from pathlib import Path

import pandas as pd
from rdfsolve import SchemaMiner

logging.basicConfig(level=logging.WARNING, format="%(message)s", force=True)
endpoint = "https://aopwiki.rdf.bigcat-bioinformatics.org/sparql"
miner = SchemaMiner(
    endpoint, graph_uris=["http://aopwiki.org/"], strategy="one-shot",
    counts=False, enrich=True, examples_per_pattern=0, timeout=30,
)
miner.helper.enable_query_collection()
schema = miner.mine("aopwikirdf")
if miner.last_report.completion_state != "complete":
    raise RuntimeError("Mining did not finish. Inspect miner.last_report before continuing.")
client = schema.client(miner.helper, max_subjects=500)

## Find pathways by name

The search looks in labels and titles, ignoring case. We ask for titles and stressor references; other fields are not loaded.

In [2]:
AOP = client.model("AdverseOutcomePathway")

with client.step("Find AOP titles containing thyroid"):
    thyroid_aops = client.search(AOP, "thyroid", fields=["title", "c54571"])

print(f"{len(thyroid_aops)} matching pathways")
client.table(thyroid_aops, ["title"])

18 matching pathways


,uri,title
0,https://identifiers.org/aop/119,Inhibition of thyroid peroxidase leading to fo...
1,https://identifiers.org/aop/128,Kidney dysfunction by decreased thyroid hormone
2,https://identifiers.org/aop/152,Interference with thyroid serum binding protei...
3,https://identifiers.org/aop/162,Enhanced hepatic clearance of thyroid hormones...
4,https://identifiers.org/aop/271,Inhibition of thyroid peroxidase leading to im...
5,https://identifiers.org/aop/300,Thyroid Receptor Antagonism and Subsequent Adv...
6,https://identifiers.org/aop/366,Competitive binding to thyroid hormone carrier...
7,https://identifiers.org/aop/367,Competitive binding to thyroid hormone carrier...
8,https://identifiers.org/aop/393,AOP for thyroid disorder caused by triphenyl p...
9,https://identifiers.org/aop/402,Thyroid peroxidase (TPO) inhibition leads to p...


Each result is an instance of the generated AOP class. You can read its fields directly. A field can contain several values.

In [3]:
aop = thyroid_aops[0]
print(aop.uri)
aop.title

https://identifiers.org/aop/119


['Inhibition of thyroid peroxidase leading to follicular cell adenomas and carcinomas (in rat and mouse)']

## What can we follow?

The schema lists fields that lead to other record types. This is a guide to available links; we still need to find which links exist for our selected pathways.

In [4]:
client.links(AOP)[["field", "target"]]

,field,target
0,seealso,AdverseOutcomePathway
1,identifier,AdverseOutcomePathway
2,c54571,Stressor
3,has_adverse_outcome,KeyEvent
4,has_key_event,KeyEvent
5,has_key_event_relationship,KeyEventRelationship
6,has_molecular_initiating_event,KeyEvent
7,prop_131567,CellularOrganisms
8,page,AdverseOutcomePathway


The `c54571` field leads to a `Stressor`. The [AOPWiki RDF data model](https://journals.sagepub.com/doi/full/10.1089/aivt.2021.0010) documents that annotation. We follow it and keep each source-to-stressor link in the session.

In [5]:
Stressor = client.model("Stressor")

with client.step("Read stressors annotated to the matching AOPs"):
    stressors = client.follow(
        thyroid_aops, "c54571", Stressor,
        fields=["title", "description", "has_chemical_entity", "page"],
    )

print(f"{len(stressors)} stressors; {sum(not a.c54571 for a in thyroid_aops)} AOPs returned no stressor link")
client.table(stressors, ["title", "description"])

24 stressors; 12 AOPs returned no stressor link


,uri,title,description
0,https://identifiers.org/aop.stressor/133,Phenobarbital,
1,https://identifiers.org/aop.stressor/148,Polychlorinated biphenyl,
2,https://identifiers.org/aop.stressor/160,Halogenated phenols,
3,https://identifiers.org/aop.stressor/170,thiazopyr,
4,https://identifiers.org/aop.stressor/171,Pyrethrins and Pyrethroids,
5,https://identifiers.org/aop.stressor/249,"2,3,7,8-tetrachlorodibenzo-p-dioxin (TCDD)",
6,https://identifiers.org/aop.stressor/25,Ethylene thiourea,
7,https://identifiers.org/aop.stressor/256,Polychlorinated dibenzodioxins,
8,https://identifiers.org/aop.stressor/257,Polybrominated diphenyl ethers,
9,https://identifiers.org/aop.stressor/258,Isoflavones,


An empty description means no text was returned for that field. The recorded link tells us which AOP a stressor belongs to; it does not supply a reason or an evidence grade. Keep the annotation's page for further review.

In [6]:
client.evidence()[["source", "field", "target"]].head(10)

,source,field,target
0,https://identifiers.org/aop/162,c54571,https://identifiers.org/aop.stressor/133
1,https://identifiers.org/aop/152,c54571,https://identifiers.org/aop.stressor/148
2,https://identifiers.org/aop/459,c54571,https://identifiers.org/aop.stressor/148
3,https://identifiers.org/aop/152,c54571,https://identifiers.org/aop.stressor/160
4,https://identifiers.org/aop/162,c54571,https://identifiers.org/aop.stressor/170
5,https://identifiers.org/aop/162,c54571,https://identifiers.org/aop.stressor/171
6,https://identifiers.org/aop/459,c54571,https://identifiers.org/aop.stressor/249
7,https://identifiers.org/aop/271,c54571,https://identifiers.org/aop.stressor/25
8,https://identifiers.org/aop/152,c54571,https://identifiers.org/aop.stressor/256
9,https://identifiers.org/aop/152,c54571,https://identifiers.org/aop.stressor/257


The full link table is saved below. The ten rows above are just a preview.

## Which stressors have chemical identifiers?

Some stressors describe groups or nonchemical factors. We keep stressors without chemical links visible instead of quietly dropping them.

In [7]:
client.links(Stressor)[["field", "target"]]

,field,target
0,identifier,Stressor
1,has_chemical_entity,ChemicalEntity
2,has_chemical_entity,CASRegistryNumber
3,ispartof,AdverseOutcomePathway
4,page,Stressor


The chemical links lead to `ChemicalEntity` records. We use `Chemical` as a short Python name for that generated class.

In [8]:
Chemical = client.model("ChemicalEntity")

with client.step("Read the identified chemicals"):
    chemicals = client.follow(
        stressors, "has_chemical_entity", Chemical,
        fields=["title", "label", "identifier", "exactmatch"],
    )

linked_ids = {iri for s in stressors for iri in s.has_chemical_entity}
unread_ids = linked_ids - {c.uri for c in chemicals}
print(f"{len(chemicals)} distinct chemicals; {len(unread_ids)} linked IRIs not returned with this type")
client.table(chemicals, ["title", "identifier"])

11 distinct chemicals; 0 linked IRIs not returned with this type


,uri,title,identifier
0,https://identifiers.org/cas/115-86-6,Triphenyl phosphate,https://identifiers.org/cas/115-86-6
1,https://identifiers.org/cas/117718-60-2,Thiazopyr,https://identifiers.org/cas/117718-60-2
2,https://identifiers.org/cas/131-55-5,"2,2',4,4'-Tetrahydroxybenzophenone",https://identifiers.org/cas/131-55-5
3,https://identifiers.org/cas/17737-65-4,Clonixin,https://identifiers.org/cas/17737-65-4
4,https://identifiers.org/cas/50-06-6,Phenobarbital,https://identifiers.org/cas/50-06-6
5,https://identifiers.org/cas/51-52-5,6-Propyl-2-thiouracil,https://identifiers.org/cas/51-52-5
6,https://identifiers.org/cas/55335-06-3,Triclopyr,https://identifiers.org/cas/55335-06-3
7,https://identifiers.org/cas/60-56-0,Methimazole,https://identifiers.org/cas/60-56-0
8,https://identifiers.org/cas/644-62-2,Meclofenamic acid,https://identifiers.org/cas/644-62-2
9,https://identifiers.org/cas/79-94-7,"3,3',5,5'-Tetrabromobisphenol A",https://identifiers.org/cas/79-94-7


In [9]:
without_chemical = [s for s in stressors if not s.has_chemical_entity]
print(f"{len(without_chemical)} of {len(stressors)} stressors returned no chemical link")
client.table(without_chemical, ["title", "page"])

13 of 24 stressors returned no chemical link


,uri,title,page
0,https://identifiers.org/aop.stressor/148,Polychlorinated biphenyl,https://identifiers.org/aop.stressor/148
1,https://identifiers.org/aop.stressor/160,Halogenated phenols,https://identifiers.org/aop.stressor/160
2,https://identifiers.org/aop.stressor/171,Pyrethrins and Pyrethroids,https://identifiers.org/aop.stressor/171
3,https://identifiers.org/aop.stressor/249,"2,3,7,8-tetrachlorodibenzo-p-dioxin (TCDD)",https://identifiers.org/aop.stressor/249
4,https://identifiers.org/aop.stressor/256,Polychlorinated dibenzodioxins,https://identifiers.org/aop.stressor/256
5,https://identifiers.org/aop.stressor/257,Polybrominated diphenyl ethers,https://identifiers.org/aop.stressor/257
6,https://identifiers.org/aop.stressor/258,Isoflavones,https://identifiers.org/aop.stressor/258
7,https://identifiers.org/aop.stressor/259,Perflourinated chemicals,https://identifiers.org/aop.stressor/259
8,https://identifiers.org/aop.stressor/260,Phthalates,https://identifiers.org/aop.stressor/260
9,https://identifiers.org/aop.stressor/264,"2,6-dinitro-p-cresol",https://identifiers.org/aop.stressor/264


## Look up one chemical

Phenobarbital appears in the results. Find it by name, then inspect its identifiers. `exactmatch` contains the identifier links published by the source; we have not independently verified each match.

In [10]:
with client.step("Look up Phenobarbital"):
    phenobarbital = client.search(
        Chemical, "Phenobarbital", fields=["title", "identifier", "exactmatch"]
    )

client.table(phenobarbital, ["title", "identifier", "exactmatch"])

,uri,title,identifier,exactmatch
0,https://identifiers.org/cas/50-06-6,Phenobarbital,https://identifiers.org/cas/50-06-6,https://identifiers.org/chebi/8069 | https://i...


## Where else do these chemicals lead?

Now follow **all** the identified chemicals back to their stressor annotations, then to the AOPs carrying those annotations. `inverse=True` means following the same recorded link backwards.

In [11]:
with client.step("Find all annotations for these chemicals"):
    chemical_stressors = client.follow(
        chemicals, "has_chemical_entity", Stressor, inverse=True, fields=["title"]
    )
    related_aops = client.follow(
        chemical_stressors, "c54571", AOP, inverse=True, fields=["title"]
    )

initial_ids = {a.uri for a in thyroid_aops}
other_aops = [a for a in related_aops if a.uri not in initial_ids]
print(f"{len(related_aops)} linked AOPs; {len(other_aops)} outside the initial title search")
client.table(other_aops, ["title"])

12 linked AOPs; 8 outside the initial title search


,uri,title
0,https://identifiers.org/aop/107,Constitutive androstane receptor activation le...
1,https://identifiers.org/aop/159,Thyroperoxidase inhibition leading to increase...
2,https://identifiers.org/aop/175,Thyroperoxidase inhibition leading to altered ...
3,https://identifiers.org/aop/363,Thyroperoxidase inhibition leading to altered ...
4,https://identifiers.org/aop/401,G protein-coupled estrogen receptor 1 (GPER) s...
5,https://identifiers.org/aop/42,Inhibition of Thyroperoxidase and Subsequent A...
6,https://identifiers.org/aop/504,SULT1E1 inhibition leading to uterine adenocar...
7,https://identifiers.org/aop/565,"17β-Hydroxysteroid dehydrogenase 2, inhibition..."


“Outside the title search” does **not** mean unrelated to the thyroid. Some titles use “thyroperoxidase”, which the word “thyroid” does not match. That gives us a useful next search rather than a reason to treat the first list as complete.

Which key events belong to these additional AOPs?

In [12]:
KeyEvent = client.model("KeyEvent")

with client.step("Read key events from the additional AOPs"):
    other_events = client.follow(other_aops, "has_key_event", KeyEvent, fields=["title"])

print(f"{len(other_events)} distinct key events")
client.table(other_events, ["title"])

35 distinct key events


,uri,title
0,https://identifiers.org/aop.events/1003,"Decreased, Triiodothyronine (T3)"
1,https://identifiers.org/aop.events/1005,"Reduced, Swimming performance"
2,https://identifiers.org/aop.events/1007,"Reduced, Anterior swim bladder inflation"
3,https://identifiers.org/aop.events/1065,"Activation, estrogen receptor alpha"
4,https://identifiers.org/aop.events/1101,"Altered, Amphibian metamorphosis"
5,https://identifiers.org/aop.events/1214,Altered gene expression specific to CAR activa...
6,https://identifiers.org/aop.events/1284,"Up Regulation, SREBF2"
7,https://identifiers.org/aop.events/1299,"Activation, AKT"
8,https://identifiers.org/aop.events/1300,"Activation, mTORC1"
9,https://identifiers.org/aop.events/1310,"Activation, PI3K"


We now have recorded routes from chemicals through stressors and AOPs to key events. These routes show **shared pathway membership**, not direct chemical–event interactions. Evidence for a causal claim must be assessed separately.

## Keep the steps and results

This table shows what ran, how many records were read, and whether HTTP fallbacks were used. A completed step means the operation returned within its limits—not that the endpoint contains a complete account of the biology.

In [13]:
client.steps()

,Step,Queries,Records read,HTTP fallbacks,Result
0,Find AOP titles containing thyroid,2,18,0,complete
1,Read stressors annotated to the matching AOPs,3,24,0,complete
2,Read the identified chemicals,3,11,0,complete
3,Look up Phenobarbital,2,1,0,complete
4,Find all annotations for these chemicals,4,25,0,complete
5,Read key events from the additional AOPs,3,35,0,complete


Save the full queries, their step numbers, matched links, schema, and software versions. Mining queries from the same helper are retained too. This is the record of how we obtained the results, not an endpoint snapshot.

In [14]:
client.save_session("thyroid-session.json")
client.table(chemicals, ["title", "identifier"]).to_csv("thyroid-chemicals.csv", index=False)
client.table(other_events, ["title"]).to_csv("thyroid-other-events.csv", index=False)
client.evidence().to_json("thyroid-links.json", orient="records", indent=2)
Path("aopwiki_shapes.ttl").write_text(schema.to_shacl(), encoding="utf-8")
miner.close()

print("Saved the session, chemical table, key-event table, links, and schema.")

Saved the session, chemical table, key-event table, links, and schema.


## What this investigation does and does not cover

We searched one named graph, started from titles and labels containing “thyroid”, and followed the displayed fields. Missing annotations and missing chemical identifiers remain visible. The client stops on request failures and its configured result limits; hidden endpoint caps can still escape detection.

Try “thyroperoxidase” next, or inspect the titles and descriptions of related key events. Do not silently broaden the question or call a shared AOP a direct effect.

Continue with [saving a small RDF subset](AOPWiki_subsets.ipynb) or [writing RDF from the chemical table](AOPWiki_table_to_RDF.ipynb).